<a href="https://colab.research.google.com/github/Azhar-Ali-ruk/Bg_Changer/blob/main/demo/VibeVoice_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable T4 GPU in Colab.")

gpu_name = torch.cuda.get_device_name(0)
print("GPU:", gpu_name)

if "T4" not in gpu_name:
    print("Warning: This notebook is optimized for T4.")

print("CUDA:", torch.version.cuda)
print("PyTorch:", torch.__version__)

GPU: Tesla T4
CUDA: 13.0
PyTorch: 2.11.0+cu130


In [22]:
%cd /content

!rm -rf /content/VibeVoice

!git clone --quiet --depth 1 \
    https://github.com/vibevoice-community/VibeVoice.git \
    /content/VibeVoice

print("Repository cloned successfully!")

assert os.path.exists(
    "/content/VibeVoice/demo/inference_from_file.py"
), "TTS inference script is missing!"

/content
Repository cloned successfully!


In [23]:
!pip install -q uv

!uv pip install --system -e /content/VibeVoice

print("Dependencies installed!")

Using Python 3.13.16 environment at: /usr
Resolved 137 packages in 46ms
Prepared 1 package in 885ms
Uninstalled 1 package in 0.44ms
Installed 1 package in 0.84ms
 ~ vibevoice==0.1.0 (from file:///content/VibeVoice)
Dependencies installed!


In [24]:
import os

model_path = "/content/models/VibeVoice-1.5B"

os.makedirs(model_path, exist_ok=True)

!HF_XET_HIGH_PERFORMANCE=1 hf download microsoft/VibeVoice-1.5B \
    --local-dir /content/models/VibeVoice-1.5B

print("Model download completed!")

Fetching 9 files: 100% 9/9 [00:00<00:00, 1504.35it/s]
/content/models/VibeVoice-1.5B
Model download completed!


In [25]:
%%writefile /content/my_transcript.txt
Speaker 1: Can I try VibeVoice with my own example?
Speaker 2: Of course! VibeVoice is open-source, built to benefit everyone. You're welcome to try it out.

Overwriting /content/my_transcript.txt


In [26]:
script_path = "/content/VibeVoice/demo/inference_from_file.py"

with open(script_path, "r", encoding="utf-8") as f:
    script = f.read()

old = 'attn_impl_primary = "flash_attention_2"'
new = 'attn_impl_primary = "sdpa"'

if old in script:
    script = script.replace(old, new, 1)
    with open(script_path, "w", encoding="utf-8") as f:
        f.write(script)
    print("T4 SDPA configuration applied!")
elif new in script:
    print("SDPA is already configured!")
else:
    raise RuntimeError("Could not locate attention configuration.")

T4 SDPA configuration applied!


In [27]:
voices_dir = "/content/VibeVoice/demo/voices"

if not os.path.isdir(voices_dir):
    raise FileNotFoundError("Voice presets directory is missing!")

voice_files = [
    f for f in os.listdir(voices_dir)
    if f.lower().endswith(".wav")
]

print("Available voices:")

for voice in sorted(voice_files):
    print(voice)

for speaker in ["en-Alice_woman", "en-Frank_man"]:
    if not any(
        os.path.splitext(f)[0].lower() == speaker.lower()
        for f in voice_files
    ):
        raise FileNotFoundError(
            f"{speaker}.wav not found. Choose an available voice instead."
        )

Available voices:
en-Alice_woman.wav
en-Carter_man.wav
en-Frank_man.wav
en-Mary_woman_bgm.wav
en-Maya_woman.wav
in-Samuel_man.wav
zh-Anchen_man_bgm.wav
zh-Bowen_man.wav
zh-Xinran_woman.wav


In [28]:
import os
import subprocess
from IPython.display import Audio, display

os.makedirs("/content/outputs", exist_ok=True)

command = [
    "python",
    "/content/VibeVoice/demo/inference_from_file.py",
    "--model_path", "/content/models/VibeVoice-1.5B",
    "--txt_path", "/content/my_transcript.txt",
    "--speaker_names", "Alice", "Frank",
    "--output_dir", "/content/outputs",
    "--dtype", "float16",
    "--device", "cuda",
]

print("Generating audio... Please wait.")

subprocess.run(command, check=True)

audio_path = "/content/outputs/my_transcript_generated.wav"

if not os.path.isfile(audio_path):
    raise FileNotFoundError("Audio generation did not produce the expected file.")

print("Audio generated successfully!")

display(Audio(filename=audio_path))

Generating audio... Please wait.
Audio generated successfully!


In [29]:
from google.colab import files

audio_path = "/content/outputs/my_transcript_generated.wav"

if os.path.exists(audio_path):
    files.download(audio_path)
else:
    print("Audio file not found. Run Step 8 first.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>